# Flowbased
In this file you will find functions that creates flow-parameters and constraints in a scalable way. 


In [1]:
include(dirname(dirname(pwd()))*"\\src\\TuLiPa.jl");
using .TuLiPa
using Dates, DataFrames, CSV, JSON, Plots, JuMP, HiGHS
include(dirname(dirname(pwd())) * raw"\demos\other\flowbased_utils.jl")
include(dirname(dirname(pwd())) * raw"\demos\other\flowbased_utils2.jl")

make_connection (generic function with 2 methods)

### Create flows
The data for this part should be replaced with data from JulES

In [2]:
# PowerA = 1.0
# PowerA_cap = 24.0
# DemandA = 10.0
# PowerB_cap = 1000000000.0
# PowerB = 100000000.0
# DemandB = 100.0  
# PowerC_cap = 1000000000.0  
# PowerC = 100.0  
# DemandC = 10.0  


PowerA = 100.0
PowerA_cap = 1000.0 / 0.024
DemandA = 140.0 / 0.024
PowerB_cap = 1000.0 / 0.024
PowerB = 200.0
DemandB = 170.0 / 0.024
PowerC_cap = 20.0  / 0.024 # Kapasitet
PowerC = 70.0 # Pris
DemandC = 140.0 / 0.024 # Volum forbruk (uelastisk)


network = Dict(
    "A" => Dict("Demand" => DemandA, "Power" => PowerA, "Power_cap" => PowerA_cap),
    "B" => Dict("Demand" => DemandB, "Power" => PowerB, "Power_cap" => PowerB_cap),
    "C" => Dict("Demand" => DemandC, "Power" => PowerC, "Power_cap" => PowerC_cap)
);

## PTDF-matrix

In [3]:
file = "./assets/3noder_enkeltlinjer_1.csv"
df_flowbased_grid = DataFrame(CSV.File(file))

Row,CnecName,emps_area0,emps_area1,A,B,C,RAM
,String15,String1,String1,Float64,Float64,Int64,Float64?
1,BORDER_A->B,A,B,0.33333,-0.33333,0,missing
2,BORDER_A->C,A,C,0.66667,0.33333,0,missing
3,BORDER_B->C,B,C,0.33333,0.66667,0,missing
4,243,A,B,0.33333,-0.33333,0,2300.0
5,465,A,C,0.6667,0.33333,0,1000.0
6,543,B,C,0.33333,0.66667,0,300.0


In [4]:
elements = create_elements(network, df_flowbased_grid);
transm = [e for e in elements if is_transmission(e)]
df_ptdf = process_ptdf_matrix(df_flowbased_grid, false)


Dict("B" => Dict("Power" => 200.0, "Power_cap" => 41666.666666666664, "Demand" => 7083.333333333333), "A" => Dict("Power" => 100.0, "Power_cap" => 41666.666666666664, "Demand" => 5833.333333333333), "C" => Dict("Power" => 70.0, "Power_cap" => 833.3333333333334, "Demand" => 5833.333333333333))

Row,CnecName,A,B,C,RAM,border
,Abstract…,Float64,Float64,Int64,Float64?,Bool
1,Transm_A->B,0.33333,-0.33333,0,missing,true
2,Transm_A->C,0.66667,0.33333,0,missing,true
3,Transm_B->C,0.33333,0.66667,0,missing,true
4,243,0.33333,-0.33333,0,2300.0,false
5,465,0.6667,0.33333,0,1000.0,false
6,543,0.33333,0.66667,0,300.0,false


In [5]:
T = DataElement
flow_based = make_flowbased(df_ptdf, transm)
elements = vcat(elements, flow_based);
modelobjects = getmodelobjects(elements);

In [6]:
mymodel = JuMP.Model(HiGHS.Optimizer)
prob = JuMP_Prob(modelobjects, mymodel)
datatime = getisoyearstart(2023)
scenariotime = getisoyearstart(1981)
prob_time = TwoTime(datatime, scenariotime)
update!(prob, prob_time)

In [7]:
solve!(prob)
print(prob.model)
print(solution_summary(prob.model, verbose = true))

-dual.(prob.model[:BalanceA]), -dual.(prob.model[:BalanceB]), -dual.(prob.model[:BalanceC])

* Solver : HiGHS

* Status
  Result count       : 1
  Termination status : OPTIMAL
  Message from the solver:
  "kHighsModelStatusOptimal"

* Candidate solution (result #1)
  Primal status      : FEASIBLE_POINT
  Dual status        : FEASIBLE_POINT
  Objective value    : 4.44000e+04
  Objective bound    : 0.00000e+00
  Relative gap       : Inf
  Dual objective value : 4.44000e+04
  Primal solution :
    FlowPowerA[1] : 4.30000e+02
    FlowPowerB[1] : 0.00000e+00
    FlowPowerC[1] : 2.00000e+01
    FlowTransm_A->B[1] : 1.53332e+02
    FlowTransm_A->C[1] : 1.36668e+02
    FlowTransm_B->A[1] : 0.00000e+00
    FlowTransm_B->C[1] : 0.00000e+00
    FlowTransm_C->A[1] : 0.00000e+00
    FlowTransm_C->B[1] : 1.66682e+01
  Dual solution :
    BalanceA[1] : -1.00000e+02
    BalanceB[1] : -1.00000e+02
    BalanceC[1] : -1.00000e+02
    FlowBased243in[1] : 0.00000e+00
    FlowBased243out[1] : 0.00000e+00
    FlowBased465in[1] : 0.00000e+00
    FlowBased465out[1] : 0.00000e+00
    FlowBased543in[1] 

([100.0], [100.0], [100.0])

In [8]:
steps = 1
problem = prob
resultobjects = collect(values(modelobjects))
numperiods_powerhorizon = 1
numperiods_hydrohorizon = 1
periodduration_power = Day(1)
t = 1
includeexogenprice=true

prices, rhstermvalues, production, consumption, hydrolevels, batterylevels, powerbalances, rhsterms, rhstermbalances, plants, plantbalances, plantarrows, demands, demandbalances, demandarrows, hydrostorages, batterystorages  = init_results(steps, problem, modelobjects, resultobjects, numperiods_powerhorizon, numperiods_hydrohorizon, periodduration_power, t, includeexogenprice);

In [9]:
println(prices)
println(production)
println(consumption)
println(powerbalances)

[100.0 100.0 100.0]
[0.0 0.0 0.6945083333333338 0.8333333333333334 6.388825 0.0 17.916666666666668 0.0 5.694508333333334]
[0.0 0.6945083333333338 6.388825 0.0 0.0 5.694508333333334]
Any[BaseBalance(Id("Balance", "B"), BaseCommodity(Id("Commodity", "Power"), SequentialHorizon(SequentialPeriods([(1, Millisecond(86400000))]), nothing)), SequentialHorizon(SequentialPeriods([(1, Millisecond(86400000))]), nothing), RHSTerm[BaseRHSTerm(Id("RHSTerm", "DemandB"), MWToGWhSeriesParam{ConstantTimeVector, ConstantTimeVector}(ConstantTimeVector(7083.333333333333), ConstantTimeVector(1.0)), false, Dict{Any, Any}())], Dict{Any, Any}()), BaseBalance(Id("Balance", "C"), BaseCommodity(Id("Commodity", "Power"), SequentialHorizon(SequentialPeriods([(1, Millisecond(86400000))]), nothing)), SequentialHorizon(SequentialPeriods([(1, Millisecond(86400000))]), nothing), RHSTerm[BaseRHSTerm(Id("RHSTerm", "DemandC"), MWToGWhSeriesParam{ConstantTimeVector, ConstantTimeVector}(ConstantTimeVector(5833.333333333333), 